# Guardian JSON -> FinBERT -> PCA32 (Colab)

This notebook is designed for **easy Colab operation**:
1. Upload one Guardian JSON file (e.g., `guardian_news_2022.json`)
2. Set config (year, output prefix, PCA dims)
3. Run all cells
4. Download parquet outputs

Outputs:
- Daily `(ticker, date)` FinBERT features with 768-d embeddings
- PCA-compressed features (default 32 dims)
- Optional article-level FinBERT output


In [1]:
# Install dependencies (Colab)
!pip -q install pyarrow transformers torch scikit-learn pandas_market_calendars tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.


In [64]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas_market_calendars as mcal

from google.colab import files

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA available: True


In [101]:
# Add this cell BEFORE the extraction logic block
from config_companies import COMPANIES
print(f'Loaded {len(COMPANIES)} companies from config')

Loaded 33 companies from config


In [102]:
# ===== User config =====
TARGET_YEAR = 2025
OUTPUT_PREFIX = f'news_finbert_{TARGET_YEAR}'
PCA_COMPONENTS = 32   # set 32 or 64
BATCH_SIZE = 32
MAX_LENGTH = 512

# If True, also save article-level FinBERT output parquet
SAVE_ARTICLE_LEVEL = True

# Optional: set explicit file name if you upload multiple files
JSON_FILE_NAME = None  # e.g., 'guardian_news_2022.json'


In [103]:
# Assuming JSON files are already uploaded in the Colab file explorer
if JSON_FILE_NAME is not None:
    json_path = Path(JSON_FILE_NAME)
else:
    # Try to infer the filename from TARGET_YEAR
    json_path = Path(f'guardian_news_{TARGET_YEAR}.json')
    if not json_path.exists():
        # Fallback to finding any json file in the current directory
        json_files = list(Path('.').glob('*.json'))
        if not json_files:
            raise ValueError('No JSON files found in the current directory. Please upload them.')
        json_path = json_files[0]

if not json_path.exists():
    raise ValueError(f'{json_path} not found. Please upload it.')

print('Using JSON file:', json_path)

with open(json_path, 'r', encoding='utf-8') as f:
    articles = json.load(f)

df = pd.DataFrame(articles)
print('Loaded rows:', len(df))
print('Columns:', df.columns.tolist())

if 'publication_date' not in df.columns:
    raise ValueError('Expected `publication_date` in JSON rows.')

df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
df = df[df['publication_date'].dt.year == TARGET_YEAR].copy()
print(f'Rows after year={TARGET_YEAR} filter:', len(df))
if df.empty:
    raise ValueError('No rows remain after year filter.')

Using JSON file: guardian_news_2025.json
Loaded rows: 1726
Columns: ['uid', 'ticker', 'webPublicationDate', 'publication_date', 'trading_date', 'guardian_id', 'type', 'sectionId', 'sectionName', 'pillarId', 'pillarName', 'webTitle', 'webUrl', 'apiUrl', 'headline', 'trailText', 'bodyText', 'text_for_finbert', 'wordcount', 'guardian_tone', 'tags', 'query', 'from_date_window', 'to_date_window', 'source', 'backfill', 'backfill_order_by']
Rows after year=2025 filter: 1726


In [104]:
# === MODIFIED: Per-ticker context isolation ===
#
# Instead of concatenating headline+trail+body and truncating at 512 tokens,
# we extract sentences around each ticker mention. This handles multi-ticker
# articles correctly: a Reuters piece mentioning AAPL, MSFT, NVDA gets three
# separate text snippets, one per ticker, each focused on that ticker.
#
# Logic per article:
#   1. If ticker appears in headline OR trailText -> use headline + trail +
#      first 3 sentences of body (the article is *about* this ticker)
#   2. Else if ticker appears in body -> extract ±N sentences around each
#      mention, dedupe overlapping windows, join (the article *mentions*
#      this ticker tangentially)
#   3. Else -> skip this (article, ticker) pair entirely
#
# Configurable params below.

import re

# ===== Context isolation config =====
CONTEXT_SENTENCES_BEFORE = 2   # sentences to grab BEFORE each ticker mention
CONTEXT_SENTENCES_AFTER = 2    # sentences to grab AFTER each ticker mention
HEADLINE_BODY_PREFIX_SENTENCES = 3  # if ticker in headline, grab first N body sentences
MAX_CONTEXT_CHARS = 2500       # hard cap before tokenizer truncation (~512 tokens)


def split_sentences(text: str) -> list[str]:
    """
    Lightweight sentence splitter. Avoids importing nltk/spacy for a Colab notebook.
    Splits on '.', '!', '?' followed by whitespace + capital letter or end.
    Good enough for news prose.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Split, keep delimiters attached to preceding sentence
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [p.strip() for p in parts if p.strip()]


import re
from functools import lru_cache

# ===== Ticker disambiguation config =====
#
# Some tickers / aliases are too ambiguous to match safely on their own.
# We require the article to ALSO mention a "context anchor" — usually
# the full company name or a distinctive product/person.
#
# Format: ticker -> set of patterns that must NOT match alone (need anchor)
AMBIGUOUS_ALIASES = {
    "AAPL": {"Apple"},        # fruit / many product names
    "META": {"Meta"},         # general english word
    "AMZN": {"Amazon"},       # river, region, mythology
    "GOOGL": {"Google"},      # often used as verb
    "AMC": {"AMC"},           # many other AMCs (American Movie Classics, etc.)
    "AERO": {"AERO"},         # generic aviation prefix
    "CCO": {"CCO"},           # also "Chief Customer Officer" abbreviation
    "ADI": {"ADI"},           # many other ADIs
    "ABEO": set(),            # ticker-only, no ambiguity expected
}

# Tickers that are real English words or too short to match standalone
# without a name anchor. We never match these as bare tokens.
TICKER_NEEDS_ANCHOR = {"AERO", "AMC", "META", "CCO", "ACES", "ALL"}


def _strip_corporate_suffixes(name: str) -> str:
    """Remove common corporate suffixes while preserving the distinctive core."""
    # Order matters: longer suffixes first
    suffixes = [
        r'\b(Group Inc\.?|Holdings Inc\.?|Company Inc\.?)\b',
        r'\b(Corporation|Incorporated|Limited|Holdings|Company|Group|Therapeutics|Pharmaceuticals|Technologies|Industries|Systems|Sciences)\b',
        r'\b(Inc\.?|Corp\.?|Ltd\.?|LLC|Plc|Co\.?|S\.?A\.?(B\.?)?( de C\.?V\.?)?)\b',
    ]
    out = name
    for pat in suffixes:
        out = re.sub(pat, '', out, flags=re.IGNORECASE)
    # Collapse whitespace and trim trailing punctuation
    out = re.sub(r'\s+', ' ', out).strip(' ,.')
    return out


def _is_short_or_common(token: str) -> bool:
    """A pattern is unsafe to match alone if it's a single short common word."""
    if len(token) <= 3:
        return True
    # Single common English word (rough heuristic — anything in title case
    # that's a dictionary word should require context)
    common_singles = {
        "Apple", "Meta", "Amazon", "Google", "Tesla", "Target",
        "Visa", "Block", "Match", "Square", "Pinterest",
    }
    return token in common_singles


@lru_cache(maxsize=512)
def _compile_patterns_for(ticker: str,
                           company_name: str,
                           aliases_tuple: tuple) -> dict:
    """
    Build a structured pattern dict. Cached because we call this once per
    article × ticker but only have ~30 unique tickers.

    Returns:
        {
          "strong":    list of patterns that match alone (full company name,
                       distinctive products, executive names)
          "weak":      list of patterns that require an anchor match elsewhere
                       in the same article
          "ticker_re": pattern matching the ticker symbol, case-sensitive
        }
    """
    aliases = list(aliases_tuple)
    strong = []
    weak = []

    # 1. Full company name (always strong — distinctive)
    if company_name:
        # Match the literal name, case-insensitive, allow trailing punctuation
        strong.append(re.compile(
            re.escape(company_name).replace(r'\ ', r'\s+'),
            re.IGNORECASE
        ))
        # Also match the suffix-stripped form if it's still distinctive (≥2 words or long enough)
        stripped = _strip_corporate_suffixes(company_name)
        if stripped and stripped != company_name:
            if ' ' in stripped or (len(stripped) >= 6 and not _is_short_or_common(stripped)):
                strong.append(re.compile(
                    re.escape(stripped).replace(r'\ ', r'\s+'),
                    re.IGNORECASE
                ))

    # 2. Aliases from config (CEOs, products, alternate names)
    for alias in aliases:
        if not alias or alias.strip() == "":
            continue
        alias = alias.strip()
        # Skip if alias is just the ticker — handled separately
        if alias == ticker:
            continue
        # Decide strong vs weak based on length/commonness
        pattern = re.compile(
            r'\b' + re.escape(alias).replace(r'\ ', r'\s+') + r'\b',
            re.IGNORECASE
        )
        if _is_short_or_common(alias):
            weak.append(pattern)
        else:
            strong.append(pattern)

    # 3. Ticker symbol — case-sensitive, must be standalone uppercase
    # This is intentionally strict: "AAPL" matches, "aapl" doesn't.
    if ticker in TICKER_NEEDS_ANCHOR:
        ticker_re = re.compile(r'\b' + re.escape(ticker) + r'\b')
        weak.append(ticker_re)
        ticker_re_returned = ticker_re
    else:
        ticker_re = re.compile(r'\b' + re.escape(ticker) + r'\b')
        # Tickers ≥4 chars are usually distinctive enough alone
        if len(ticker) >= 4:
            strong.append(ticker_re)
        else:
            weak.append(ticker_re)
        ticker_re_returned = ticker_re

    return {
        "strong": tuple(strong),
        "weak": tuple(weak),
        "ticker_re": ticker_re_returned,
    }


def build_ticker_patterns_from_config(company_record: dict) -> dict:
    """
    Build patterns from a COMPANIES record.

    Args:
        company_record: one entry from COMPANIES list, with keys
                        'ticker', 'company_name', 'entity_match_patterns'

    Returns:
        Structured pattern dict (see _compile_patterns_for).
    """
    ticker = company_record["ticker"]
    company_name = company_record.get("company_name", "")
    aliases = company_record.get("entity_match_patterns", [])
    return _compile_patterns_for(ticker, company_name, tuple(aliases))


def find_ticker_mentions(text: str, patterns: dict) -> list[tuple[int, int, str]]:
    """
    Find all ticker mentions in text.

    A "mention" qualifies if EITHER:
      - any 'strong' pattern matches anywhere in the text, OR
      - a 'weak' pattern matches AND at least one strong pattern also matches
        somewhere (anywhere) in the text (the anchor requirement).

    Returns list of (start, end, matched_text) tuples, sorted by position.
    """
    if not isinstance(text, str) or not text:
        return []

    # First pass: find all strong matches (these always count)
    strong_hits = []
    for pat in patterns["strong"]:
        for m in pat.finditer(text):
            strong_hits.append((m.start(), m.end(), m.group()))

    # If we have any strong hit, weak hits are also valid (anchor satisfied)
    weak_hits = []
    if strong_hits:
        for pat in patterns["weak"]:
            for m in pat.finditer(text):
                weak_hits.append((m.start(), m.end(), m.group()))

    # Combine and dedupe overlapping spans
    all_hits = sorted(strong_hits + weak_hits, key=lambda x: x[0])
    deduped = []
    last_end = -1
    for start, end, txt in all_hits:
        if start >= last_end:
            deduped.append((start, end, txt))
            last_end = end
    return deduped


def extract_windows(sentences: list[str], hit_indices: list[int],
                    before: int, after: int) -> str:
    """
    For each hit, take ±N sentences. Merge overlapping windows. Return joined text.
    """
    if not hit_indices:
        return ''
    # Build set of all sentence indices to include
    keep = set()
    for h in hit_indices:
        for j in range(max(0, h - before), min(len(sentences), h + after + 1)):
            keep.add(j)
    # Re-sort and join
    keep_sorted = sorted(keep)
    return ' '.join(sentences[j] for j in keep_sorted)


# Build a ticker -> company_record lookup once at notebook start
COMPANIES_BY_TICKER = {c["ticker"]: c for c in COMPANIES}


def extract_ticker_context(row, ticker_col: str = 'ticker') -> str:
    """Replacement for make_text(). Returns ticker-focused snippet from one article."""
    # Honor upstream pre-extracted text
    preferred = row.get('text_for_finbert', None)
    if isinstance(preferred, str) and preferred.strip():
        return preferred.strip()[:MAX_CONTEXT_CHARS]

    ticker = row.get(ticker_col, None)
    if not isinstance(ticker, str) or ticker not in COMPANIES_BY_TICKER:
        return ''

    patterns = build_ticker_patterns_from_config(COMPANIES_BY_TICKER[ticker])

    headline = row.get('headline', '') or ''
    trail = row.get('trailText', '') or ''
    body = row.get('bodyText', '') or ''

    # Case 1: ticker in headline or trailText -> article is *about* this ticker
    in_metadata = bool(find_ticker_mentions(headline + ' ' + trail, patterns))

    body_sents = split_sentences(body)

    if in_metadata:
        prefix = ' '.join([headline.strip(), trail.strip()]).strip()
        body_prefix = ' '.join(body_sents[:HEADLINE_BODY_PREFIX_SENTENCES])
        return (prefix + ' ' + body_prefix).strip()[:MAX_CONTEXT_CHARS]

    # Case 2: ticker only in body -> find mention sentences and extract ±N windows
    hit_sent_indices = []
    for i, sent in enumerate(body_sents):
        if find_ticker_mentions(sent, patterns):
            hit_sent_indices.append(i)

    # If weak-only matches in body but strong elsewhere in the article, find_ticker_mentions
    # at sentence level might miss the anchor. Re-check at article level for safety.
    if not hit_sent_indices:
        article_hits = find_ticker_mentions(body, patterns)
        if article_hits:
            # Map character positions back to sentence indices
            cum_pos = 0
            hit_starts = [h[0] for h in article_hits]
            for i, sent in enumerate(body_sents):
                sent_start = body.find(sent, cum_pos)
                sent_end = sent_start + len(sent)
                if any(sent_start <= hs < sent_end for hs in hit_starts):
                    hit_sent_indices.append(i)
                cum_pos = sent_end

    if hit_sent_indices:
        windowed = extract_windows(body_sents, hit_sent_indices,
                                   CONTEXT_SENTENCES_BEFORE,
                                   CONTEXT_SENTENCES_AFTER)
        out = (headline.strip() + ' ' + windowed).strip() if headline else windowed
        return out[:MAX_CONTEXT_CHARS]

    return ''


# Apply the new extractor
df['text_input'] = df.apply(extract_ticker_context, axis=1)

# Drop rows where no ticker context was found (article didn't actually
# mention the ticker, or mention was in metadata only)
before_drop = len(df)
df = df[df['text_input'].str.len() > 0].copy()
after_drop = len(df)
print(f'Context extraction: kept {after_drop} / {before_drop} (article, ticker) rows')
print(f'Dropped {before_drop - after_drop} where no ticker mention was found in text')
print(f'Mean text length: {df["text_input"].str.len().mean():.0f} chars')
print(f'Median text length: {df["text_input"].str.len().median():.0f} chars')
print(f'Max text length: {df["text_input"].str.len().max()} chars (cap = {MAX_CONTEXT_CHARS})')

Context extraction: kept 1726 / 1726 (article, ticker) rows
Dropped 0 where no ticker mention was found in text
Mean text length: 2486 chars
Median text length: 2500 chars
Max text length: 2500 chars (cap = 2500)


In [105]:
# trading_date in your JSON is often null, so fallback to publication_date.
if 'trading_date' in df.columns:
    td = pd.to_datetime(df['trading_date'], errors='coerce')
else:
    td = pd.Series(pd.NaT, index=df.index)

df['event_date'] = td.fillna(df['publication_date'])

# Shift weekends to next business day; then align to NYSE schedule.
df['event_date'] = pd.to_datetime(df['event_date']).dt.normalize()
weekend = df['event_date'].dt.dayofweek >= 5
df.loc[weekend, 'event_date'] = df.loc[weekend, 'event_date'] + pd.offsets.BDay(1)

print('Rows ready for FinBERT:', len(df))
print('\n--- Sample Text ---')
sample_text = df['text_input'].iloc[0]
print(f"Length in characters: {len(sample_text)}")
print(f"Content preview: {sample_text[:500]}...")

Rows ready for FinBERT: 1726

--- Sample Text ---
Length in characters: 2500
Content preview: UK demands ability to access Apple users’ encrypted data Expert says government has ‘lit the blue touch paper on a truly enormous fight’ as it challenges firm’s privacy stance The UK government has demanded that Apple creates a backdoor in its encrypted cloud service, in a confrontation that challenges the US tech firm’s avowed stance on protecting user privacy. The Washington Post reported on Friday that the Home Office had issued a “technical capability notice” under the Investigatory Powers A...


In [106]:
print('Rows ready for FinBERT:', len(df))
print('\n--- Sample Text ---')
sample_text = df['text_input'].iloc[0]
print(f"Length in characters: {len(sample_text)}")
print(f"Content preview: {sample_text}")

Rows ready for FinBERT: 1726

--- Sample Text ---
Length in characters: 2500
Content preview: UK demands ability to access Apple users’ encrypted data Expert says government has ‘lit the blue touch paper on a truly enormous fight’ as it challenges firm’s privacy stance The UK government has demanded that Apple creates a backdoor in its encrypted cloud service, in a confrontation that challenges the US tech firm’s avowed stance on protecting user privacy. The Washington Post reported on Friday that the Home Office had issued a “technical capability notice” under the Investigatory Powers Act (IPA), which requires companies to assist law enforcement in providing evidence. The demand, issued last month, relates to Apple’s Advanced Data Protection (ADP) service, which heavily encrypts personal data uploaded and stored remotely in Apple’s cloud servers, according to the Post, which said this was a “blanket” request that applied to any Apple user worldwide. The ADP service uses end-to-end enc

In [107]:
# Diagnostic: per-ticker match rate
diagnostics = []
for ticker in df['ticker'].unique():
    sub = df[df['ticker'] == ticker]
    n_total = len(sub)
    n_kept = (sub['text_input'].str.len() > 0).sum()
    diagnostics.append({
        'ticker': ticker,
        'total': n_total,
        'kept': n_kept,
        'kept_pct': 100 * n_kept / n_total if n_total else 0,
    })
diag_df = pd.DataFrame(diagnostics).sort_values('kept_pct')
print(diag_df.to_string(index=False))

# Tickers with <30% kept rate likely need more aliases in config
low = diag_df[diag_df['kept_pct'] < 30]
print(f"\n{len(low)} tickers with low match rate — review entity_match_patterns:")
print(low['ticker'].tolist())

ticker  total  kept  kept_pct
  AAPL    198   198     100.0
  ADBE     10    10     100.0
   ADP     40    40     100.0
   AMD     34    34     100.0
  AMGN      2     2     100.0
  AMZN    199   199     100.0
  AVGO      6     6     100.0
 CMCSA     31    31     100.0
  COST      2     2     100.0
  CSCO      1     1     100.0
  GILD      3     3     100.0
 GOOGL    199   199     100.0
  INTU      2     2     100.0
  LRCX      1     1     100.0
  META    198   198     100.0
  MSFT    198   198     100.0
    MU      6     6     100.0
  NFLX    114   114     100.0
  NVDA    165   165     100.0
  ORCL     74    74     100.0
   PEP     12    12     100.0
  QCOM     14    14     100.0
  REGN      2     2     100.0
  SBUX     27    27     100.0
  TSLA    186   186     100.0
   TXN      1     1     100.0
  VRTX      1     1     100.0

0 tickers with low match rate — review entity_match_patterns:
[]


In [108]:
# FinBERT inference: 3-way probs + 768-d [CLS] embedding
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'ProsusAI/finbert'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True)
model = model.to(device)
model.eval()

texts = df['text_input'].tolist()
all_probs = []
all_emb = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='FinBERT batches'):
        batch_texts = texts[i:i+BATCH_SIZE]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(**enc)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        # Last hidden state: [batch, seq_len, 768], take CLS token
        cls = out.hidden_states[-1][:, 0, :].cpu().numpy()

        all_probs.append(probs)
        all_emb.append(cls)

probs = np.vstack(all_probs)
emb = np.vstack(all_emb)

# ProsusAI/finbert label order is usually [positive, negative, neutral].
# We map by label2id to avoid assumptions.
id2label = model.config.id2label
label2idx = {v.lower(): int(k) for k, v in id2label.items()}
required = ['positive', 'negative', 'neutral']
for r in required:
    if r not in label2idx:
        raise ValueError(f'Label {r} not found in model labels: {label2idx}')

df['sentiment_pos'] = probs[:, label2idx['positive']]
df['sentiment_neg'] = probs[:, label2idx['negative']]
df['sentiment_neu'] = probs[:, label2idx['neutral']]

for j in range(emb.shape[1]):
    df[f'emb_{j}'] = emb[:, j].astype(np.float32)

print('Article-level inference complete.')
print(df[['sentiment_pos','sentiment_neg','sentiment_neu']].head())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT batches:   0%|          | 0/54 [00:00<?, ?it/s]

/tmp/ipykernel_18853/47165372.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'emb_{j}'] = emb[:, j].astype(np.float32)
/tmp/ipykernel_18853/47165372.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'emb_{j}'] = emb[:, j].astype(np.float32)
/tmp/ipykernel_18853/47165372.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented

Article-level inference complete.
   sentiment_pos  sentiment_neg  sentiment_neu
0       0.053963       0.437384       0.508653
1       0.059374       0.037257       0.903369
2       0.016368       0.747187       0.236445
3       0.031850       0.484442       0.483708
4       0.031715       0.479611       0.488674


/tmp/ipykernel_18853/47165372.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'emb_{j}'] = emb[:, j].astype(np.float32)
/tmp/ipykernel_18853/47165372.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'emb_{j}'] = emb[:, j].astype(np.float32)
/tmp/ipykernel_18853/47165372.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented

In [109]:
# Aggregate to daily (ticker, date), then create full ticker-date grid with has_news flag
df['date'] = pd.to_datetime(df['event_date']).dt.normalize()

emb_cols = [f'emb_{i}' for i in range(768)]
agg_cols = ['sentiment_pos', 'sentiment_neu', 'sentiment_neg'] + emb_cols

daily_news = (
    df.groupby(['ticker', 'date'], as_index=False)
      .agg({**{c: 'mean' for c in agg_cols}, 'text_input': 'count'})
      .rename(columns={'text_input': 'n_articles'})
)
daily_news['has_news'] = 1
daily_news['mean_tone'] = daily_news['sentiment_pos'] - daily_news['sentiment_neg']

# Full NYSE date grid for the year and observed tickers in uploaded file.
nyse = mcal.get_calendar('NYSE')
sched = nyse.schedule(start_date=f'{TARGET_YEAR}-01-01', end_date=f'{TARGET_YEAR}-12-31')
trading_days = pd.to_datetime(sched.index).normalize()
tickers = sorted(df['ticker'].dropna().unique().tolist())

grid = pd.MultiIndex.from_product([tickers, trading_days], names=['ticker', 'date']).to_frame(index=False)
daily = grid.merge(daily_news, on=['ticker', 'date'], how='left')

fill_zero_cols = agg_cols + ['n_articles', 'has_news', 'mean_tone']
for c in fill_zero_cols:
    daily[c] = daily[c].fillna(0.0)

daily['n_articles'] = daily['n_articles'].astype(np.int32)
daily['has_news'] = daily['has_news'].astype(np.int8)

print('Daily rows:', len(daily))
print('Tickers:', len(tickers), 'Trading days:', len(trading_days))
print('News-day share:', float((daily['has_news'] == 1).mean()))

/tmp/ipykernel_18853/3469590289.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['date'] = pd.to_datetime(df['event_date']).dt.normalize()


Daily rows: 6750
Tickers: 27 Trading days: 250
News-day share: 0.1482962962962963


In [110]:
# PCA compression to 32/64 dims (fit on news days only)
if PCA_COMPONENTS not in [32, 64]:
    raise ValueError('PCA_COMPONENTS should be 32 or 64 for this workflow.')

fit_mask = daily['has_news'] == 1
if fit_mask.sum() < PCA_COMPONENTS:
    raise ValueError(f'Not enough news rows ({fit_mask.sum()}) for PCA-{PCA_COMPONENTS}.')

X_fit = daily.loc[fit_mask, emb_cols].to_numpy(dtype=np.float64)
X_all = daily[emb_cols].to_numpy(dtype=np.float64)

pca = PCA(n_components=PCA_COMPONENTS, svd_solver='full', random_state=42)
pca.fit(X_fit)
Z = pca.transform(X_all)

pca_cols = [f'news_pca_{i:02d}' for i in range(PCA_COMPONENTS)]
pca_df = pd.DataFrame(Z, columns=pca_cols)

daily_pca = pd.concat([
    daily[['ticker', 'date', 'sentiment_pos', 'sentiment_neu', 'sentiment_neg', 'n_articles', 'has_news', 'mean_tone']].reset_index(drop=True),
    pca_df.reset_index(drop=True)
], axis=1)

print(f'PCA-{PCA_COMPONENTS} explained variance: {pca.explained_variance_ratio_.sum():.4f}')

PCA-32 explained variance: 0.8822


In [111]:
# Save parquet outputs and download
out_daily_768 = Path(f'{OUTPUT_PREFIX}_daily_768.parquet')
out_daily_pca = Path(f'{OUTPUT_PREFIX}_daily_pca{PCA_COMPONENTS}.parquet')
out_article = Path(f'{OUTPUT_PREFIX}_articles_finbert.parquet')

daily.to_parquet(out_daily_768, index=False)
daily_pca.to_parquet(out_daily_pca, index=False)

print('Saved:', out_daily_768)
print('Saved:', out_daily_pca)

if SAVE_ARTICLE_LEVEL:
    keep_article_cols = [
        'ticker', 'publication_date', 'event_date', 'date',
        'sentiment_pos', 'sentiment_neu', 'sentiment_neg',
        'text_input'
    ] + emb_cols
    df[keep_article_cols].to_parquet(out_article, index=False)
    print('Saved:', out_article)

# Download files to local machine
files.download(str(out_daily_768))
files.download(str(out_daily_pca))
if SAVE_ARTICLE_LEVEL:
    files.download(str(out_article))

Saved: news_finbert_2025_daily_768.parquet
Saved: news_finbert_2025_daily_pca32.parquet
Saved: news_finbert_2025_articles_finbert.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [112]:
print(f"Finished parsing file for year: {TARGET_YEAR}")

Finished parsing file for year: 2025


In [114]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Create the INDENG242 folder in MyDrive
drive_dir = Path('/content/drive/MyDrive/INDENG242temp')
drive_dir.mkdir(parents=True, exist_ok=True)
print(f"Folder ready at: {drive_dir}")

# Define the files to copy based on previous cell outputs
files_to_copy = [out_daily_768, out_daily_pca]
if SAVE_ARTICLE_LEVEL:
    files_to_copy.append(out_article)

# Copy the files to the Google Drive folder
for f in files_to_copy:
    if f.exists():
        dest = drive_dir / f.name
        shutil.copy(f, dest)
        print(f"Copied {f.name} to {dest}")
    else:
        print(f"Warning: {f.name} not found.")

Mounted at /content/drive
Folder ready at: /content/drive/MyDrive/INDENG242temp
Copied news_finbert_2025_daily_768.parquet to /content/drive/MyDrive/INDENG242temp/news_finbert_2025_daily_768.parquet
Copied news_finbert_2025_daily_pca32.parquet to /content/drive/MyDrive/INDENG242temp/news_finbert_2025_daily_pca32.parquet
Copied news_finbert_2025_articles_finbert.parquet to /content/drive/MyDrive/INDENG242temp/news_finbert_2025_articles_finbert.parquet


In [115]:
from pathlib import Path
import shutil

# Target Google Drive folder
drive_dir = Path('/content/drive/MyDrive/INDENG242temp')
drive_dir.mkdir(parents=True, exist_ok=True)

# Find all parquet files in the current directory
local_dir = Path('.')
parquet_files = list(local_dir.glob('*.parquet'))

print(f"Found {len(parquet_files)} parquet files. Copying to Drive...")

for f in parquet_files:
    dest = drive_dir / f.name
    shutil.copy(f, dest)
    print(f"Copied {f.name}")

print("All files copied successfully!")

Found 24 parquet files. Copying to Drive...
Copied news_finbert_2020_daily_pca32.parquet
Copied news_finbert_2019_daily_768.parquet
Copied news_finbert_2021_daily_pca32.parquet
Copied news_finbert_2018_daily_768.parquet
Copied news_finbert_2022_daily_768.parquet
Copied news_finbert_2019_daily_pca32.parquet
Copied news_finbert_2024_daily_pca32.parquet
Copied news_finbert_2025_articles_finbert.parquet
Copied news_finbert_2022_articles_finbert.parquet
Copied news_finbert_2025_daily_pca32.parquet
Copied news_finbert_2019_articles_finbert.parquet
Copied news_finbert_2023_articles_finbert.parquet
Copied news_finbert_2020_articles_finbert.parquet
Copied news_finbert_2023_daily_768.parquet
Copied news_finbert_2018_articles_finbert.parquet
Copied news_finbert_2021_articles_finbert.parquet
Copied news_finbert_2022_daily_pca32.parquet
Copied news_finbert_2025_daily_768.parquet
Copied news_finbert_2020_daily_768.parquet
Copied news_finbert_2023_daily_pca32.parquet
Copied news_finbert_2018_daily_pc